# Wildfire Prediction

## Table of content

- Read data

- Exploratory data analysis
    - Data structure
    - Class balance
    - Fires by Months Across All Years
    - Geographical Info
    - FWI distribution
    - Fire Class by time period

- ML Model Training
    - Load train/validate/test split
    - Initialize Models
        - Random Forest
        - Logistic Regression
    - Hyperparameter Tuning 
        -  Randomized Search + `TimeSeriesSplit`
    - Model Review
    - Validate Hyperparameter tuning (Validation set)
        - F1 Score
        - Recall
        - AUC
        - Classification Report
    - Repeat Hyperparameter and Validation (...if required)
        - Refine hyperparameters
    - Testing (Test set)
        - F1 Score
        - Recall
        - AUC
        - Classification Report
- Outputs
    - Generates the various images, tables, and outputs required for write-up 

## Read data

### Libraries

In [1]:
import utils.datasets_utils as du
import ml_models.ml_utils as mu
import data_io.ml_io as mlio
import src.pipelines.viirs_pipeline as vp
import src.data_io.ukgrid_loader as ul
import ml_models.ml_visualisations as mlvis
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import contextily as ctx
import numpy as np
import re
import os 
import time

from typing import Any
from pathlib import Path
from joblib import dump
from scripts.s00_set_parameters import PARAMETERS, ML_CLEANED_MODELS_NAMES
# Set to True if plots need to be saved to disk to add to report
SAVE_PLOTS = False


### Load data

The .csv files containing all the processed data to run this notebook were too large to either commit to GitHub or to add as Zip files via Turnitin. Therefore, the code block below takes the processed data from the local machine, transforms them to parquet files. The parquet files are committed to GitHub, and the load section is changed from reading .csv to reading parquet.

This allows users cloning the repository to be able to run the notebook and replicate the results with the same data. Please note that the models hyperparameters combinations were also committed to GitHub for demo purposes

In [8]:
# Base path/location of data
path_base = PARAMETERS['DATA_DIR']/"MLInputs"
# CSV fnames
# Input files to load
files_to_load = {"resnet_default_weights":  "2026-07-12_ml_input.csv",
                 "resnet_layer4_finetuned": "2026-08-10_ml_input_layer4_finetuned.csv"}
# Generate full paths 
files_to_load = {k: os.path.join(path_base, v) for (k, v) in files_to_load.items()}
# Spefic data types requirements
string_cols = {"composite_key": "string", "composite_key_y": "string", "bridge_composite_key_y": "string"}
date_cols   = ["date_y", "date"]

# List files from data directory to check if Parquet files already exist or not
parquet_files = [p for p in list(path_base.iterdir()) if str(p).endswith('parquet')]
parquet_loaded = False
while not parquet_loaded:
    # If the parquet files are not available, transform the .csv to parquet
    if not parquet_files:
        # Load data 
        df_resnet_default   = pd.read_csv(files_to_load["resnet_default_weights"], dtype  = string_cols, parse_dates=date_cols) #type: ignore
        df_resnet_finetuned = pd.read_csv(files_to_load["resnet_layer4_finetuned"], dtype = string_cols, parse_dates=date_cols) #type: ignore

        # Change float 64 to float 32 to save further space as compression alone did not meet github size limits
        default_float_cols = df_resnet_default.select_dtypes(include=["float64"]).columns
        df_resnet_default[default_float_cols] = df_resnet_default[default_float_cols].astype("float32")
        finetuned_float_cols = df_resnet_finetuned.select_dtypes(include=["float64"]).columns
        df_resnet_finetuned[finetuned_float_cols] = df_resnet_finetuned[finetuned_float_cols].astype("float32")

        # save as parquet
        df_resnet_default.to_parquet(os.path.join(path_base, "2026-07-12_ml_input.parquet"), compression = "brotli", index = False)
        df_resnet_finetuned.to_parquet(os.path.join(path_base, "2026-08-10_ml_input_layer4_finetuned.parquet"), compression = "brotli", index = False)
        continue
    df_resnet_default   = pd.read_parquet(os.path.join(path_base, "2026-07-12_ml_input.parquet"))
    df_resnet_finetuned = pd.read_parquet(os.path.join(path_base, "2026-08-10_ml_input_layer4_finetuned.parquet"))
    parquet_loaded = True


In [ ]:
# Base path/location of data
path_base = PARAMETERS['DATA_DIR']/"MLInputs"
# List files from data directory to check if Parquet files area
# Input files to load
files_to_load = {"resnet_default_weights":  "2026-07-12_ml_input.csv",
                 "resnet_layer4_finetuned": "2026-08-10_ml_input_layer4_finetuned.csv"}
# Generate full paths 
files_to_load = {k: os.path.join(path_base, v) for (k, v) in files_to_load.items()}
# Spefic data types requirements
string_cols = {"composite_key": "string", "composite_key_y": "string", "bridge_composite_key_y": "string"}
date_cols   = ["date_y", "date"]
# Load data 
df_resnet_default   = pd.read_csv(files_to_load["resnet_default_weights"], dtype  = string_cols, parse_dates=date_cols) #type: ignore
df_resnet_finetuned = pd.read_csv(files_to_load["resnet_layer4_finetuned"], dtype = string_cols, parse_dates=date_cols) #type: ignore


## Exploratory data analysis

### Data structure

Ensure that the structure of both data frames is the same. The only difference should be the values in the feature columns - everything else should be equal

In [9]:
# Validate structure is the same
print("Structure Validation: ")
print(f"- Columns match:    {df_resnet_finetuned.columns.equals(df_resnet_default.columns)}")
print(f"- Data types match: {df_resnet_finetuned.dtypes.equals(df_resnet_default.dtypes)}")
print(f"- Data Shape match: {df_resnet_finetuned.shape == df_resnet_default.shape}")

# Validate that the only data differences are found in feature columns, all other data points should be the same between data frames 
feat_cols               = [c for c in df_resnet_default.columns if re.search("^feat_", c)]
df_minus_feat_default   = df_resnet_default.drop(columns=feat_cols)
df_minus_feat_finetuned = df_resnet_finetuned.drop(columns=feat_cols)
# Validate data matches
print("\nData Validation:")
try:
    pd.testing.assert_frame_equal(df_minus_feat_default,  df_minus_feat_finetuned)
    print("- Non-feature data match: True")
except AssertionError as e:
    print("- Non-feature data match: False")
    print(e)

Structure Validation: 
- Columns match:    True
- Data types match: True
- Data Shape match: True

Data Validation:
- Non-feature data match: True


Columns and data types for reference:

In [10]:
col_names = df_resnet_default.columns
for pos, dt in enumerate(df_resnet_default.dtypes):
    print(f"{col_names[pos]}: \t\t\t{dt}")
  

composite_key_y: 			string
grid_id_y: 			int64
sample_type_y: 			str
date_y: 			datetime64[us]
fire_lbl_y: 			bool
bridge_composite_key_y: 			string
grid_id: 			float32
x_coord: 			float32
y_coord: 			float32
geometry: 			str
date: 			datetime64[us]
viirs_n: 			float32
frp_max: 			float32
frp_mean: 			float32
fire_lbl: 			bool
fwi_max: 			float32
fwi_mean: 			float32
composite_key: 			string
feat_000: 			float32
feat_001: 			float32
feat_002: 			float32
feat_003: 			float32
feat_004: 			float32
feat_005: 			float32
feat_006: 			float32
feat_007: 			float32
feat_008: 			float32
feat_009: 			float32
feat_010: 			float32
feat_011: 			float32
feat_012: 			float32
feat_013: 			float32
feat_014: 			float32
feat_015: 			float32
feat_016: 			float32
feat_017: 			float32
feat_018: 			float32
feat_019: 			float32
feat_020: 			float32
feat_021: 			float32
feat_022: 			float32
feat_023: 			float32
feat_024: 			float32
feat_025: 			float32
feat_026: 			float32
feat_027: 			float32
feat_028: 			floa

The full dataset contains the columns below. All the columns that end with suffix `y` refer to be values to be predicted. These are the values that the model needs to predict. The rest of the columns contain the data to train the model. The training data (non `y` columns) are from a t-1 from value to predict. 

In [ ]:
# Check assumptions for all datasets
for _, row in df_resnet_default.iterrows():
    row: Any
    days_diff = (row.date_y - row.date).days
    if days_diff != 1:
        print('❌ Date assumption of t-1 not met!')
print(f"✅ t-1 assumption validated\n  [{df_resnet_default.shape[0]} rows checked]")

### Class balance

The sampling procedure implemented a 2:1 no-fire to fire target ratio when selecting the samples. The reason of this is that no-fire events are much more common than fire events. However, using the real distribution would have produced an extremely imbalanced dataset. To mitigate this, a 2:1 target ratio was implemented to maintain the frequency property but reducing the class imbalance to a more manageable state, as done by previous studies. 

The complete details of the sampling procedure are in:
- Functions: `src/sampling/sampling_functions.py` 
- Pipeline: `src/pipelines/sampling_pipeline.py`

In [ ]:
print("=== Class Balance ===")
df_class_report = df_resnet_default.copy()

df_class_report['year'] = pd.to_datetime(df_class_report['date']).dt.year

df_class_report_summary = (df_class_report
                           .groupby(['year', 'fire_lbl_y'])
                           .size()
                           .unstack(fill_value=0)
                           .rename(columns={True: 'Fire', False: 'No Fire'}))

df_class_report_summary['Total'] = df_class_report_summary['Fire'] + df_class_report_summary['No Fire']

print(df_class_report_summary)



### Fires by Month Across All Years


In [ ]:
df_fire = df_resnet_default[df_resnet_default['fire_lbl_y'] == True].copy()
monthly_fire = (df_fire
                .assign(month=lambda x: pd.to_datetime(x['date']).dt.month)
                .groupby('month')
                .size()
                )
plt.figure(figsize=(10, 5))

plt.bar(monthly_fire.index,
        monthly_fire.values, #type: ignore
        color = 'tomato')

plt.xticks(range(1, 13),
           ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
            'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])

plt.xlabel('Month')
plt.ylabel('Number of fire observations')
plt.title('Monthly distribution of fire observations')
plt.tight_layout()
if SAVE_PLOTS:
        plt.savefig(f"outputs/plots/fire_count_by_month.png",
                        dpi=300,
                        bbox_inches="tight")
plt.show()


### Geographical Info

In [ ]:
# Load UK grid
uk_grid = ul.load_uk_grid(PARAMETERS['DATA_DIR'],PARAMETERS['SP_FILENAME'], PARAMETERS['CRS'])
uk_grid = ul.load_uk_grid(PARAMETERS['DATA_DIR'],PARAMETERS['SP_FILENAME'], PARAMETERS['CRS'])
uk_grid.head()


In [ ]:
fig_ukmap, ax_ukmap = plt.subplots(figsize = (12,12,))

uk_grid.boundary.plot(ax=ax_ukmap,
                      markersize=0.05,
                      color="red",
                      alpha=0.3,
                      edgecolor="none",
                      linewidth = 0.4)
ctx.add_basemap(ax=ax_ukmap, crs = PARAMETERS['CRS'])
ax_ukmap.set_axis_off()
if SAVE_PLOTS:
    fig_ukmap.savefig(f"outputs/plots/grids_over_uk_map.png",
                    dpi=300,
                    bbox_inches="tight")

plt.show()


In [ ]:
# Load VIIRS data
dict_viirs = vp.load_viirs_main(PARAMETERS['YEAR_FILTER'], PARAMETERS['DATA_DIR'], PARAMETERS['CRS'])
df_viirs = dict_viirs['df_viirs']
df_viirs.head()


In [ ]:

fig_viirs, ax_viirs = plt.subplots(figsize=(12,12,))
df_viirs.plot(ax=ax_viirs,
              markersize=2,
              color="firebrick",
              alpha=0.35,
              edgecolor="none")
ctx.add_basemap(ax=ax_viirs, crs = PARAMETERS['CRS'])
ax_viirs.set_axis_off()
if SAVE_PLOTS:
    fig_viirs.savefig(f"outputs/plots/viirs_geographical_dist_over_uk_map.png",
                    dpi=300,
                    bbox_inches="tight")

plt.show()


### FWI Distribution

#### Tabular

The class imbalance in the predictor variable is evident, with only 1,816 fire observations vs 36,388 no-fire observations (see `df_fwi_descriptives` below).

The descriptive statistics show that the fire class, overall, has higher FWI values than the no fire, with the median of fire observations (*M = 2.55*) almost 3 times higher than the no-fire (*M = 0.91*) counterpart. It is worth nothing, however, that the max value of no-fire is greater than the fire (more than 2sd above the max). This is the case for 7/36388 of the no-fire observations, which indicates these are rare observations (See FWI - BLOCK 2).

In [ ]:
# FWI - BLOCK 1 - Descriptives
df_fwi_descriptives = df_resnet_default[['fwi_mean', 'fire_lbl']].groupby('fire_lbl').describe()
df_fwi_descriptives

In [ ]:
# FWI - BLOCK 2 - FWI value max review
# Count of no fire observations above the max of Fire
fwi_fire_lbl_max = df_fwi_descriptives.loc[True, ("fwi_mean", "max")]
df_resnet_default[ (df_resnet_default['fire_lbl'] == False) & (df_resnet_default['fwi_mean'] > fwi_fire_lbl_max)].shape[0]

#### Visualization

To visually compare the distribution of FWI values between fire and no-fire observations a histogram of raw counts is not appropriate given the large class imbalance. The dataset contains many more no-fire than fire observations, therefore, the no-fire class dominates the histogram, making it difficult to compare the two classes

In [ ]:
ax = sns.histplot(data=df_resnet_default,
                  x="fwi_mean",
                  hue = 'fire_lbl',
                  bins = 50,
                  edgecolor = "white",
                  palette = {False: '#4C72B0', True:'#D55E00'})
ax.set_xlabel("Fire Weather Index (Mean)")
ax.set_ylabel("Count")
ax.set_title("Distribution of Fire Weather Index")
ax.legend_.set_title(title="Fire Label") # type: ignore


Instead, comparing the **distribution** of each class provides a more informative visualization of the FWI data. By normalizing each class independently, the underlying distributions can be compared without being obscured by the class imbalance, making the class differences easier to observe.

The `common_norm` parameter is set to `False` to ensure that each class is normalized independently. By default, the histogram is normalized across both classes, meaning that the combined area under the two distributions would sum to 1, which makes visual comparisons hard to communicate as the majority class would overtake the visual space. Instead, with `common_norm` set to `False` the function normalizes each class separately, so the area under each class distribution sums to 1. This allows to visually compare the fire and no-fire observations distributions independently of class imbalance. 


In [ ]:
ax = sns.histplot(data = df_resnet_default,
                  x = "fwi_mean",
                  hue = "fire_lbl",
                  bins = 50,
                  stat = "density",
                  common_norm = False,
                  edgecolor = "white",
                  palette={False: "#2F5597",
                           True: "#F00707"  })

ax.set_xlabel("Fire Weather Index (Mean)")
ax.set_ylabel("Density")
ax.set_title("Distribution of Fire Weather Index")
ax.legend_.set_title(title="Fire Label") # type: ignore
if SAVE_PLOTS:
    plt.savefig(f"outputs/plots/fwi_density_hist.png",
                    dpi=300,
                    bbox_inches="tight")

### Data by Class over time period

In [ ]:
# Helper function to create stacked bar chart to show split by temporal bins and fire, no-fire observations
def summarise_obs_by_time(df_in:pd.DataFrame, temporal_split = 'year') -> dict:
    barchart_data = {}
    for row in df_in.itertuples():
        row: Any
        temporal_string = du.get_temporal_label(row.date_y, temporal_split)
        fire_lbl        = row.fire_lbl_y

        if temporal_string not in barchart_data:
            barchart_data[temporal_string] = {"fire": 0, "no_fire": 0}

        if fire_lbl:
            barchart_data[temporal_string]["fire"] += 1
        else:
            barchart_data[temporal_string]["no_fire"] += 1
    return barchart_data


Review how many observations per year

In [ ]:
df_plot = pd.DataFrame(summarise_obs_by_time(df_resnet_default, 'year'))
plt.bar(df_plot.columns,
        df_plot.loc['no_fire'],
        color = 'dodgerblue',
        label="No Fire")

plt.bar(df_plot.columns,
        df_plot.loc['fire'],
        bottom = df_plot.loc['no_fire'],
        color = 'red',
        label="Fire")


plt.xlabel("Year")
plt.ylabel("Number of Observations")
plt.title("Yearly Distribution of Sampled Fire and Non-Fire Observations")

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
df_plot

## ML Model Training

### Load Train/Test split

In [ ]:
# =============
# SPLIT DATA
# =============
# Load data split to ensure sets are consistent across the pipeline
df_split = pd.read_csv(path_base/"2026-08-10_train_val_test_split.csv", dtype={'composite_key': str})

df_train_keys      = df_split.loc[df_split['split_category'] == 'train',    'composite_key']
df_validation_keys = df_split.loc[df_split['split_category'] == 'validation', 'composite_key']
df_test_keys       = df_split.loc[df_split['split_category'] == 'test',     'composite_key']

df_train       = df_resnet_default[df_resnet_default['composite_key'].isin(df_train_keys)].sort_values("date").reset_index(drop = True)     
df_validation  = df_resnet_default[df_resnet_default['composite_key'].isin(df_validation_keys)].sort_values("date").reset_index(drop = True)     
df_test        = df_resnet_default[df_resnet_default['composite_key'].isin(df_test_keys)].sort_values("date").reset_index(drop = True)

df_train_ft       = df_resnet_finetuned[df_resnet_finetuned['composite_key'].isin(df_train_keys)].sort_values("date").reset_index(drop = True)     
df_validation_ft  = df_resnet_finetuned[df_resnet_finetuned['composite_key'].isin(df_validation_keys)].sort_values("date").reset_index(drop = True)     
df_test_ft        = df_resnet_finetuned[df_resnet_finetuned['composite_key'].isin(df_test_keys)].sort_values("date").reset_index(drop = True)

## REMINDER - All feature column names are already saved in:
feat_cols = feat_cols

# Validate that the temporal split for both datasets is the same apart from Sentinel-2 data
# This allows for a single set of y values to be used instead of duplicating work
validation_pairs = [("Training",   df_train, df_train_ft),
                    ("Validation", df_validation, df_validation_ft),
                    ("Test",       df_test, df_test_ft)]

for name, df_a, df_b in validation_pairs:
    print(f"\n{name} data validation:")
    if df_a.shape[0] == 0 or df_b.shape[0] == 0:
        raise ValueError (f"[{name}] is an emtpy dataframe - please review")
    print(f"Rows in {name}: Default [{df_a.shape[0]}], FT [{df_b.shape[0]}]")
    try:
        pd.testing.assert_frame_equal(df_a.drop(columns=feat_cols), df_b.drop(columns=feat_cols))
        print("- Non-feature data match: True")

    except AssertionError as e:
        print("- Non-feature data match: False")
        raise e

print("\n==== Check date order ====")
print("Train chronological:", df_train["date"].is_monotonic_increasing)
print("Validation chronological:", df_validation["date"].is_monotonic_increasing)
print("Test chronological:", df_test["date"].is_monotonic_increasing)



In [ ]:
# Hybrid Model columns
hybrid_cols = feat_cols + ['fwi_mean']
# =============
# Y VALUES
# =============
# Split Y labels by default and training to ensure correctness and consitency
y_train      = df_train["fire_lbl_y"]
y_validation = df_validation["fire_lbl_y"]
y_test       = df_test["fire_lbl_y"]

# =============
# X VALUES
# =============
# FWI can be taken from default or fine tuned, no difference in data as shown in code blocks above
X_train_fwi       = df_train.loc[:, ["fwi_mean"]]
X_validation_fwi  = df_validation.loc[:,  ["fwi_mean"]]
X_test_fwi        = df_test.loc[:,  ["fwi_mean"]]

# Sentinel-2 Default ResNet18 Weights
X_train_sent2_def      = df_train.loc[:, feat_cols]
X_validation_sent2_def = df_validation.loc[:, feat_cols]
X_test_sent2_def       = df_test.loc[:, feat_cols]
# Hybrid, Sentinel-2 Default ResNet18 Weights
X_train_hybrid_def      = df_train.loc[:, hybrid_cols]
X_validation_hybrid_def = df_validation.loc[:, hybrid_cols]
X_test_hybrid_def       = df_test.loc[:, hybrid_cols]

# Sentinel-2 fineTuned ResNet18 Weights
X_train_sent2_ft       = df_train_ft.loc[:, feat_cols]
X_validation_sent2_ft  = df_validation_ft.loc[:, feat_cols]
X_test_sent2_ft        = df_test_ft.loc[:, feat_cols]
# Hybrid, Sentinel-2 fineTuned ResNet18 Weights
X_train_hybrid_ft      = df_train_ft.loc[:, hybrid_cols]
X_validation_hybrid_ft = df_validation_ft.loc[:, hybrid_cols]
X_test_hybrid_ft       = df_test_ft.loc[:, hybrid_cols]

# =================================
# TRAINNING DATA
# =================================
training_predictor_data = {'fwi': X_train_fwi,
                           
                           'sentinel_default': X_train_sent2_def,
                           'hybrid_default'  : X_train_hybrid_def,

                           'sentinel_fineTuned': X_train_sent2_ft,
                           'hybrid_fineTuned'  : X_train_hybrid_ft}


### Initialize Models

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

random_state = 42

# Initialise models
models = {'random_forest': RandomForestClassifier(random_state=random_state,n_jobs =-1),
          'logistic_reg':  Pipeline([("scaler", StandardScaler()),
                                        ("lr", LogisticRegression(random_state=random_state,
                                                                  max_iter = 1000))])}


### Hyperparameter Tuning

In [ ]:
# Hyperparameters
params = {'random_forest': {
          'criterion': ['gini', 'entropy'], 
          'n_estimators': [100, 250, 500],  
          'max_features': ['log2', 'sqrt', 0.3,0.5], 
          'max_depth': [None, 10, 20, 30], 
          'min_samples_split': [2, 5, 10], 
          'min_samples_leaf': [1, 2, 4],  
          'class_weight': [None, 'balanced']},

        'logistic_reg': {'lr__solver': ['lbfgs', 'liblinear'],
                         'lr__C': [100,10,1,0.1,0.01],
                         'lr__class_weight': [None, 'balanced']
                         }}

Load existing trained models to avoid running hyperparameter tuning on all the models again. 

**If hyperparameter tuning is required, irrespective of existing models, set `update_all_models` to `True`**

In [ ]:
update_all_models = False

load_existing_models = mlio.get_best_models()
existing_models = [k for k, _ in load_existing_models.items() if not update_all_models]
existing_models

In [ ]:

# Initialise summary results object
hyperparams_results = {}
prev_df_cv_checker = None

# Loop over each required dataset for training
for d_name, d in training_predictor_data.items():
    # Loop over each model 
    for m_name, model in models.items():
        model_name_i = f"{m_name}_{d_name}"
        if model_name_i in existing_models:
            print(f"\n\t\t >>>> Skipping: [{model_name_i}] <<<<")
            continue

        print(f"\n\t\t >>>> Processing: [{model_name_i}] <<<<")
        print("Starting search...")
        start = time.time()
        random_search_obj = mu.random_search_cv(model = model,
                                                 X = d,
                                                 y = y_train,
                                                 dates = df_train['date'],
                                                 param_distributions=params[m_name],
                                                 n_iter = 40,
                                                 n_splits = 5,
                                                 scoring = {"f1": "f1",
                                                            "roc_auc": "roc_auc",
                                                            "precision": "precision",
                                                            "recall": "recall"})
        print(f"Search took {time.time() - start:.2f} seconds")

        df_cv_checker = mu.create_cv_splits_summary(random_search_obj, d, y_train, df_train['date'])
        if prev_df_cv_checker is None:
            prev_df_cv_checker = df_cv_checker.copy()
        else:
            pd.testing.assert_frame_equal(df_cv_checker,prev_df_cv_checker)
            prev_df_cv_checker = df_cv_checker

        hyperparams_results[model_name_i] = random_search_obj
        best_idx = random_search_obj.best_index_
        best_auc = random_search_obj.cv_results_["mean_test_roc_auc"][best_idx]
        dump(random_search_obj,
             f"data/MLModels/{model_name_i}.joblib")
        print(f"Best F1: {random_search_obj.best_score_:.3f}")
        print(f"Best AUC: {best_auc:.3f}")
        print("\nBest Parameters:")
        for k, v in random_search_obj.best_params_.items():
            print(f"  {k}: {v}")


In [ ]:
if not update_all_models:
    hyperparams_results = load_existing_models

folds_metrics = mu.create_cv_splits_summary(hyperparams_results['logistic_reg_fwi'], 
                                            X_train_fwi,
                                            y_train,
                                            df_train['date'])

folds_metrics

### Review best models

In [ ]:
# Load models one more time, for inspection and to ensure that the most up to date data is used
best_models          = mlio.get_best_models()
df_models_by_fold    = mu.get_best_model_folds(best_models, ML_CLEANED_MODELS_NAMES)
best_models

In [ ]:
# Print the best hyperparameter configuration
for k, v in best_models.items():
    print(k)
    print(v.best_params_)

Plot performance by folds

In [ ]:
mlvis.plot_model_perf_by_folds(df_by_folds = df_models_by_fold, metric = "f1", model_family = "Logistic Regression",save_plot = SAVE_PLOTS)
mlvis.plot_model_perf_by_folds(df_by_folds = df_models_by_fold, metric = "f1", model_family = "Random Forest", save_plot =  SAVE_PLOTS)


Get summary metrics across folds

In [ ]:

df_models_summarised = df_models_by_fold.groupby('model')[["f1", "precision","recall", "roc_auc"]].agg(["mean", "std"]).round(3)
df_models_summarised.columns = [f"{metric}_{stat}" for metric, stat in df_models_summarised.columns]

df_models_summarised = df_models_summarised.reset_index()
df_models_summarised

In [ ]:
hyperparams_results

### Validate Hyperparameter tuning

In [ ]:
validation_predictor_data = {'fwi': X_validation_fwi,
                           
                              'sentinel_default': X_validation_sent2_def,
                              'hybrid_default'  : X_validation_hybrid_def,
    
                              'sentinel_fineTuned': X_validation_sent2_ft,
                              'hybrid_fineTuned'  : X_validation_hybrid_ft}

In [ ]:
validation_results_lst = mu.evaluate_models(best_models, validation_predictor_data, y_validation)
validation_summary     = mu.summarise_model_results(validation_results_lst,
                                                    ML_CLEANED_MODELS_NAMES)
validation_summary


### Test with Best Hyperparameters on unseen data

In [ ]:
testing_predictor_data = {'fwi': X_test_fwi,
                           
                          'sentinel_default': X_test_sent2_def,
                          'hybrid_default'  : X_test_hybrid_def,

                          'sentinel_fineTuned': X_test_sent2_ft,
                          'hybrid_fineTuned'  : X_test_hybrid_ft}
    

In [ ]:
test_results_lst = mu.evaluate_models(best_models, testing_predictor_data, y_test)
df_test_results  = mu.summarise_model_results(test_results_lst,
                                                    ML_CLEANED_MODELS_NAMES)
df_test_results

#### Outputs

##### Prepare data for Plotting general results 



In [ ]:
def prepare_plot_df(df_in: pd.DataFrame, model_name:str, model_order: list[str]) -> pd.DataFrame:
    """
    Prepares the data to plot the general results for a specified model
    the function takes the data frame, model name and order of varaibles
    It filters the data to the relevant model, and orders the rows based on the provided model order

    Parameters:
    df_in (pd.DataFrame): Input data frame containing test results
    model_name (str): The prefix for the model to filter
    model_order (list[str]): The desired order of models for plotting

    Returns:
    pd.DataFrame: A filtered and ordered data frame ready for plotting
    """
    # Filter the data frame for the specified model name
    df_plot = df_in[df_in["Model"].str.startswith(model_name)].copy()
     
    # Generate model order for plotting
    plot_order = [f"{model_name} - {m}" for m in model_order]

    # Set the 'model' column as a categorical type with the specified order
    df_plot['Model'] = pd.Categorical(df_plot['Model'], categories=plot_order, ordered=True)    
    return df_plot.sort_values(by='Model')

In [ ]:
# Pre define order of rows for both Randomg Forest and Logistic Regression models
model_order = ["FWI", "Sentinel (Default)", "Sentinel (Fine-Tuned)", "Hybrid (Default)", "Hybrid (Fine-Tuned)"]


df_lr = prepare_plot_df(df_test_results, "Logistic Regression", model_order)
df_rf = prepare_plot_df(df_test_results, "Random Forest", model_order)
df_lr


##### Score comparison across models

In [ ]:
df_lr

In [ ]:
def plot_main_results(ax, 
                      df_for_plot: pd.DataFrame, 
                      bar_width: float = 0.25,
                      show_legend: bool = True):

    # Plot parameters
    text_bar_font_size = 9
    text_bar_color = "white"

    # X axis labels
    labels = ["FWI",
              "Sentinel (Default)", 
              "Sentinel (Fine-Tuned)", 
              "Hybrid (Default)", 
              "Hybrid (Fine-Tuned)"]

    # Get the order of the models for plotting 
    x = np.arange(len(labels))

    # Extract the variables to plot 
    f1_scores = df_for_plot['F1'].values
    auc_scores = df_for_plot['ROC_AUC'].values
    fire_recall = df_for_plot['Recall'].values


    # Draw the bars
    bars_f1 =  ax.bar(x - bar_width, f1_scores, width=bar_width, label='F1 Score')
    bars_auc = ax.bar(x, auc_scores, width=bar_width, label='AUC Score')
    bars_recall = ax.bar(x + bar_width, fire_recall, width=bar_width, label='Fire Recall')

    ax.set_xticks(x)
    ax.set_xticklabels(labels)

    # Add value labels inside the bars
    # Add values inside each bar
    ax.bar_label(bars_f1,
                labels=[f"{v:.3f}" for v in f1_scores],
                label_type="center",
                fontsize=text_bar_font_size,
                color=text_bar_color)

    ax.bar_label(bars_auc,
                labels=[f"{v:.3f}" for v in auc_scores],
                label_type="center",
                fontsize=text_bar_font_size,
                color=text_bar_color)

    ax.bar_label(bars_recall,
                labels=[f"{v:.3f}" for v in fire_recall],
                label_type="center",
                fontsize=text_bar_font_size,
                color=text_bar_color)


    # Find the max value for each metric
    for bars, values in [(bars_f1, f1_scores),
                         (bars_auc, auc_scores),
                         (bars_recall, fire_recall)]:
        # Find the index of the max of the metric
        idx = np.argmax(values) # type: ignore
        # Find the corresponding bar object
        bar = bars[idx]
        # Add a star marker above the bar to indicate the max value
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height(),
                "*",
                ha="center",
                va="bottom",
                fontsize=14,
                fontweight="bold",
                color="black")

    # Set title and labels
    ax.set_ylim(0, 1)
    ax.set_ylabel("Score")
    if show_legend:
        ax.legend()

# Prepare path to save the plot
if SAVE_PLOTS:
        Path("outputs/plots").mkdir(parents=True, exist_ok=True)

# Create and save images
fig, ax = plt.subplots(figsize=(10, 6))
plot_main_results(ax, df_lr, 0.25, "Logistic Regression") # type: ignore
if SAVE_PLOTS:
        fig.savefig("outputs/plots/logistic_regression_comparison.png",
                dpi=300,
                bbox_inches="tight")

fig, ax = plt.subplots(figsize=(10, 6))
plot_main_results(ax, df_rf, 0.25, "Random Forest") # type: ignore
if SAVE_PLOTS:
        fig.savefig("outputs/plots/random_forest_comparison.png",
                dpi=300,
                bbox_inches="tight")

plt.show()

#### Performance by Metrics

In [ ]:
print(df_test_results.columns.tolist())

In [ ]:
df_by_f1 = (df_test_results[['Model', 'F1', 'Recall', 'Precision', 'ROC_AUC']]
            .sort_values('F1', ascending=False)
            .rename(columns={"F1": "F1",
                             "Recall": "Fire Recall",
                             "Precision": "Fire Precision",
                             "ROC_AUC": "ROC AUC"})
            .round({"F1": 3,
                    "Fire Recall": 3,
                    "Fire Precision": 3,
                    "ROC AUC": 3})
            .reset_index(drop=True))
# Clean model names for presentation 
df_by_f1['Model'] = (df_by_f1['Model']
                     .str.replace("logistic_reg", "LR", regex=False)
                     .str.replace("random_forest", "RF", regex=False)
                     .str.replace("sentinel_default", "Sentinel", regex=False)
                     .str.replace("sentinel_fineTuned", "Sentinel FT", regex=False)
                     .str.replace("hybrid_default", "Hybrid", regex=False)
                     .str.replace("hybrid_fineTuned", "Hybrid FT", regex=False)
                     .str.replace("fwi", "FWI", regex=False)
                     .str.replace("_", " ", regex=False))
df_by_f1


In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

plt.figure(figsize=(9, 7))

for m_name, search in load_existing_models.items():

    # Only plot Hybrid models
    if "hybrid" not in m_name.lower():
        continue

    # Find corresponding test dataset
    data_finder = re.sub(r"random_forest_|logistic_reg_","",m_name)
    X_test      = testing_predictor_data[data_finder]
    # Best model selected during RandomizedSearchCV
    model = search.best_estimator_
    # Predict probability of Fire
    y_prob = model.predict_proba(X_test)[:, 1]
    # ROC curve and AUC
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    # Clean model name for plot
    lbl_name = ML_CLEANED_MODELS_NAMES.get(m_name, m_name)

    plt.plot(fpr,
             tpr,
             lw=1,
             label=f"{lbl_name} (AUC={auc:.3f})")

# Random classifier
plt.plot([0, 1],
         [0, 1],
         'k--',
         label="Random")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(fontsize=8)
plt.grid(alpha=0.3)

plt.tight_layout()
if SAVE_PLOTS:
    plt.savefig("outputs/plots/roc_curves_hybrid_models.png",
                dpi=300,
                bbox_inches="tight")

plt.show()

In [ ]:
def compare_finetuned_models(df_in: pd.DataFrame, comp_metric: str) -> pd.DataFrame:

    df_transform = df_in[df_in['Model'].str.contains('Sentinel|Hybrid')].copy()
    df_transform = df_transform[['Model', comp_metric]]

    #Extract predictor type
    df_transform['predictor'] = df_transform['Model'].apply(lambda x: "Sentinel" if "Sentinel" in x else "Hybrid")
    # Extract CNN whether fine tuned or not
    df_transform['cnn_type'] = df_transform['Model'].apply(lambda x: "Fine-Tuned" if "Fine-Tuned" in x else "Default")
    # Extract ML model
    df_transform['ml_model'] = df_transform['Model'].apply(lambda x: "Random Forest" if "Random Forest" in x else "Logistic Regression")
    # Comvine model name
    df_transform['model_name'] = df_transform['ml_model'] + " - " + df_transform['predictor']
    # Comparison df
    comparison = (df_transform.pivot(index='model_name', columns='cnn_type', values=comp_metric).reset_index().round(3))
    comparison['Improvement'] = comparison['Fine-Tuned'] - comparison['Default']
    return comparison.sort_values('Improvement', ascending=False).reset_index(drop=True)
print("_______________________")
print("F1 Comparison")
print("_______________________")
print(compare_finetuned_models(df_test_results, 'F1'))
print("_______________________")
print("Recall Comparison")
print("_______________________")
print(compare_finetuned_models(df_test_results, 'Recall'))
